# Agent的上下文工程

## overview 

在构建 Agent 或者 任何LLM应用的时候，难点一般在于使其足够可靠。虽然一些应用在原型的时候可以正常工作，但是经常会在真实使用中失败

为什么 Agent 会任务失败？

当 Agent 任务失败时，通常是因为Agent内部的LLM调用采取了错误的行为，没有按照预期的去做。 LLM 失败的原因来两种：
1. 底层LLM能力不足
2. 没有传递正确的 上下文 给LLM

一般来说都是第二个惹的祸，导致Agent变得不可靠

上下文工程（Context Engineering）就是 为 `LLM` 提供正确的 **信息、工具，并且按照正确的格式完成任务** 的 **上下文**。 “正确”的上下文是Agent变得更加可靠的首要任务，Langchain的Agent抽象设计，为了促进上下文工程



## Context 是什么？

> 关于上下文的相关概念

上下文工程是构建动态系统的实践，这些系统能够提供 **合适的信息** 和 **工具** ，以正确的 **格式** ，使 AI 应用能够完成任务。上下文可以通过两个关键维度来描述，一个是可变性、一个是按照生命周期



1. 可变性
   - Static Context：静态上下文，在执行期间不变的数据，比如说用户元数据、数据库连接、工具
   - Dynamic Context：动态上下文，在执行期间不断变化的数据，比如说对话历史记录、中间结果、工具调用观察结果
2. 生命周期
   - Runtime Context：运行时上下文，仅限在单个运行或调用期间存在的
   - Cross-conversation：跨对话上下文，在多个对话中持续存在的数据

> 运行时上下文（Runtime Context）指的是本地的上下文，也就是代码运行时所需的数据和依赖项，运行时上下文不是指：
> 1. LLM 的上下文、也就是传递给大模型的所有提示数据（消息列表、skills、mcp等等数据）
> 2. 上下文窗口，也就是可以传递给模型的最大tokens数量
>
> Runtime Context 是一种依赖注入形式，可以用于优化传入给LLM的上下文。它允许在运行时而不是硬编码的方式向 工具 和 节点 提供依赖项（比如说数据库连接、用户ID），可以使用运行时上下文中的用户元数据来获取用户偏好，并且将他们写入LLM的上下文窗口


langgraph提供了三种管理上下文的方式，结合了可变性和生命周期两个维度

| Context type 上下文类型 | Description 描述 | Mutability 可变性 | Lifetime 生命周期 | Access method 访问方法 |
| --- | --- | --- | --- | --- |
| [**Static runtime context 静态运行时上下文**](https://docs.langchain.com/oss/python/concepts/context#static-runtime-context) | User metadata, tools, db connections passed at startup   在启动时传递的用户元数据、工具、数据库连接 | Static 静态 | Single run 单次运行 |   传入 `context` 参数到 `invoke` / `stream` |
| [**Dynamic runtime context (state)   动态运行时上下文（状态）**](https://docs.langchain.com/oss/python/concepts/context#dynamic-runtime-context-state) | Mutable data that evolves during a single run   在单次运行期间演变的可变数据 | Dynamic 动态 | Single run 单次运行 | LangGraph state object LangGraph 状态对象 |
| [**Dynamic cross-conversation context (store)   动态跨对话上下文（存储）**](https://docs.langchain.com/oss/python/concepts/context#dynamic-cross-conversation-context-store) | Persistent data shared across conversations   跨对话持久共享数据 | Dynamic 动态 | Cross-conversation 跨对话 | LangGraph store LangGraph 存储 |


【一句话来说】

> langgraph里面有三种管理上下文的方式，一种是静态的上下文，在启动时传递用户元数据、工具、数据库连接，这些在单次运行期间是不会变化的，通过 context 参数传递给 invoke / stream；第二种是单次运行的State（状态），在单次运行期间可变的数据，比如说消息列表，或者自定义的字段（权限变化）；第三种是跨对话的Store，跨对话持久共享数据。

### Static Runtime Context

静态运行时上下文代表不可变数据，如用户元数据、工具和数据库连接，这些数据通过 context 参数在运行开始时传递给应用程序 / invoke / stream 。这些数据在执行过程中不会改变。

我们可以通过 `@dataclass`、Pydantic、TypedDict 来定义 Context 的格式

### Dynamic Runtime Context -- State

动态运行时上下文表示在单次运行期间可以演变的可变数据，并通过 **LangGraph State** 对象进行管理。这包括 **对话历史记录、中间结果以及从工具或 LLM 输出中派生的值** 。在 LangGraph 中，状态对象在运行期间充当短期记忆。 State -> 短期记忆